# 03 — Customer Funnel Analysis

This notebook constructs the chronological customer funnel: product view, cart, and purchase.

## Coverage
- Funnel stage definitions
- Event sequencing and chronology
- Unique customer counts
- Step and overall conversion
- Drop-off analysis
- Channel, device, and campaign breakdowns
- Funnel metrics export

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "processed" / "ecommerce_funnel_events_clean.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError("Run the real-data pipeline first")

df = pd.read_csv(DATA_PATH)
df["event_timestamp"] = pd.to_datetime(df["event_timestamp"], utc=True, errors="coerce")
STAGES = ["product_view", "cart", "purchase"]
counts = df.groupby("stage")["user_id"].nunique().reindex(STAGES)

rows = []
for i, stage in enumerate(STAGES):
    users = int(counts[stage])
    previous = users if i == 0 else int(counts[STAGES[i - 1]])
    rows.append({
        "stage": stage,
        "users": users,
        "step_conversion_rate": 1.0 if i == 0 else users / previous if previous else 0.0,
        "overall_conversion_rate": users / int(counts.iloc[0]) if counts.iloc[0] else 0.0,
    })

funnel = pd.DataFrame(rows)
display(funnel)
funnel.to_csv(ROOT / "reports" / "funnel_summary.csv", index=False)


,stage,users,step_conversion_rate,overall_conversion_rate
0,product_view,1404179,1.000000,1.000000
1,cart,37722,0.026864,0.026864
2,purchase,11719,0.310668,0.008346
